In [2]:
!pip install -q transformers accelerate bitsandbytes sentence-transformers faiss-cpu pypdf

In [69]:
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Free the old 3B model from T4 VRAM first
if "model" in globals():
    del model
    gc.collect()
    torch.cuda.empty_cache()

# Best all-rounder for JSON + critique + writing on a 15GB T4:
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
# Alternative for punchier marketing copy (uses ~8.2 GB VRAM):
# MODEL_ID = "mistralai/Mistral-Nemo-Instruct-2407"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True, # saves an extra ~0.4 GB VRAM
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map={"": 0},
)

print(f"Loaded {MODEL_ID} | VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB / 15.0 GB")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Loaded Qwen/Qwen2.5-7B-Instruct | VRAM used: 7.86 GB / 15.0 GB


In [77]:
import json
import re
import torch
from sentence_transformers import SentenceTransformer

# Keep embeddings on CPU to reserve 100% of remaining T4 VRAM for Qwen-7B
embedder = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")

def generate(prompt, system_prompt="You are an expert direct-response copywriter.", max_new_tokens=450, temperature=0.65):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    input_ids = inputs["input_ids"]

    do_sample = temperature > 0.0
    gen_kwargs = {
        "max_new_tokens": max_new_tokens,
        "do_sample": do_sample,
        "pad_token_id": tokenizer.eos_token_id,
    }
    if do_sample:
        gen_kwargs["temperature"] = temperature
        gen_kwargs["top_p"] = 0.9
        gen_kwargs["repetition_penalty"] = 1.08

    with torch.no_grad():
        outputs = model.generate(**inputs, **gen_kwargs)

    return tokenizer.decode(outputs[0][input_ids.shape[1]:], skip_special_tokens=True).strip()


def generate_json(prompt, max_new_tokens=700, retries=2):
    sys_prompt = "You are a strict JSON API. Return ONLY valid, parseable JSON with no markdown fences or commentary."
    raw = ""
    for attempt in range(retries + 1):
        raw = generate(
            prompt,
            system_prompt=sys_prompt,
            max_new_tokens=max_new_tokens,
            temperature=0.1 if attempt == 0 else 0.35
        )
        raw_clean = re.sub(r"```json\s*|\s*```", "", raw).strip()
        # Auto-repair common LLM double-brace typos inside arrays
        raw_clean = re.sub(r"\{\s*\{", "{", raw_clean)
        raw_clean = re.sub(r"\}\s*\}", "}", raw_clean)
        # Remove trailing commas before ] or }
        raw_clean = re.sub(r",\s*([\]}])", r"\1", raw_clean)

        match = re.search(r"\{.*\}", raw_clean, re.DOTALL)
        if match:
            try:
                return json.loads(match.group(0))
            except json.JSONDecodeError:
                continue

    print(f"⚠️ JSON parse warning after retries; using fallback critique.\nRaw:\n{raw[:200]}...")
    return {
        "buyer_gut_reaction": "Could not parse detailed critique this round.",
        "new_issues": [],
        "hook_strength": 2,
        "believability_and_proof": 2,
        "clarity_and_punch": 2
    }

print("✓ Helpers & CPU embedder synced with Qwen2.5-7B-Instruct.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✓ Helpers & CPU embedder synced with Qwen2.5-7B-Instruct.


In [72]:
import re
import faiss
import numpy as np

SWIPE_FILE = [
    # Hook / Problem patterns
    "You don't have a traffic problem. You have a leaky bucket—and every dollar you spend on ads is pouring straight onto the floor.",
    "Most teams waste 6 hours every Monday stitching together spreadsheets that nobody reads by Wednesday.",
    "Stop guessing which landing page headline works. Test three variations in ten minutes and let real buyer clicks decide.",
    # Mechanism / Solution patterns
    "Instead of manual follow-ups, our engine flags stalled deals the exact hour a prospect opens your proposal twice.",
    "Built for solo founders who hate complex dashboards: connect your Stripe account once and get a plain-English cash forecast every morning at 7 AM.",
    # Proof / CTA patterns
    "Over 1,400 independent studios cut invoice chasing by 68% in their first 30 days. Start your 14-day free trial—no credit card required.",
    "See it on your own data in under three minutes. Book a 15-minute walkthrough or launch a sandbox workspace right now."
]

swipe_embeddings = embedder.encode(SWIPE_FILE, normalize_embeddings=True).astype("float32")
swipe_index = faiss.IndexFlatIP(swipe_embeddings.shape[1])
swipe_index.add(swipe_embeddings)

def retrieve_swipe(query, k=2):
    qvec = embedder.encode([query], normalize_embeddings=True).astype("float32")
    _, indices = swipe_index.search(qvec, min(k, len(SWIPE_FILE)))
    return [SWIPE_FILE[i] for i in indices[0]]

print(f"✓ Marketing swipe-file index ready ({swipe_index.ntotal} high-converting examples).")

✓ Marketing swipe-file index ready (7 high-converting examples).


In [80]:
AI_HYPE_WORDS = [
    "unleash", "unlock", "elevate", "revolutionize", "game-changer",
    "game changer", "cutting-edge", "seamless", "seamlessly", "delve",
    "skyrocket", "supercharge", "empower", "tapestry", "holistic",
    "in today's fast-paced", "look no further", "next-level", "imagine "
]

def scan_marketing_copy(markdown_text, brief):
    lower = markdown_text.lower()
    words = re.findall(r"\b[a-z']+\b", lower)
    sentences = [s.strip() for s in re.split(r"[.!?]+", markdown_text) if len(s.strip().split()) > 2]
    sections = re.split(r"^##\s+.+$", markdown_text, flags=re.MULTILINE)[1:]

    found_hype = [w.strip() for w in AI_HYPE_WORDS if w in lower]
    you_count = sum(1 for w in words if w in ("you", "your", "yours", "you're"))
    we_count = sum(1 for w in words if w in ("we", "our", "ours", "us"))
    avg_sentence_len = round(len(words) / max(1, len(sentences)), 1)

    # 1. Check that ALL numbers in proof_metric (e.g., "4.5" and "900") appear
    required_nums = re.findall(r"\d+(?:\.\d+)?", brief.get("proof_metric", ""))
    missing_proof_nums = [n for n in required_nums if n not in markdown_text]

    # 2. Check for duplicate CTA in early sections (sections 1 & 2 should NOT have trial/CTA pitches)
    cta_leaked_early = any(
        any(kw in s.lower() for kw in ("free trial", "14-day", "credit card", "sign up", "get started"))
        for s in sections[:-1]
    )

    # 3. Detect repeated 2-word phrases across different sections
    sec_bigrams = []
    for s in sections:
        toks = [w for w in re.findall(r"\b[a-z]{4,}\b", s.lower()) if w not in ("your", "with", "that", "from", "into", "just", "time", "hours")]
        sec_bigrams.append(set(zip(toks, toks[1:])))
    repeated_phrases = set()
    for i in range(len(sec_bigrams)):
        for j in range(i + 1, len(sec_bigrams)):
            repeated_phrases.update(" ".join(bg) for bg in (sec_bigrams[i] & sec_bigrams[j]))

    return {
        "hype_words": found_hype,
        "you_count": you_count,
        "we_count": we_count,
        "self_centered": we_count > you_count,
        "avg_sentence_len": avg_sentence_len,
        "too_wordy": avg_sentence_len > 17.0,
        "missing_proof_nums": missing_proof_nums,
        "cta_leaked_early": cta_leaked_early,
        "repeated_phrases": sorted(repeated_phrases),
        "word_count": len(words),
    }


def draft_marketing_piece(brief, tone="direct, punchy, conversational"):
    beats = [
        {
            "stage": "The Hook & Problem",
            "goal": f"Call out {brief['target_audience']} and name their exact frustration: {brief['core_pain']}.",
            "constraint": "Focus ONLY on the problem. Do NOT mention the product name or any free trial/CTA yet."
        },
        {
            "stage": "The Cost of Inaction & Mechanism",
            "goal": f"Explain how {brief['product_name']} solves it: {brief['mechanism']}.",
            "constraint": "Do NOT repeat phrases from Section 1. Do NOT include a call-to-action or free trial offer at the end."
        },
        {
            "stage": "Proof & Call to Action",
            "goal": f"State the concrete proof as a fact ({brief['proof_metric']}) and end with the CTA: {brief['cta']}.",
            "constraint": "Do NOT start with 'Imagine'. State the numbers directly and end with the CTA once."
        }
    ]

    print(f"🎯 Drafting PAS copy for: {brief['product_name']}")
    drafted = []
    prior_text = ""

    for beat in beats:
        swipe_ctx = "\n".join(f"- {s}" for s in retrieve_swipe(beat["goal"], k=2))
        avoid_repeat = f"\nALREADY WRITTEN (do NOT reuse these phrases):\n{prior_text}\n" if prior_text else ""

        prompt = f"""Write one short, high-converting section of landing page copy.
Match the punchy rhythm of the SWIPE EXAMPLES without copying their product details.

SWIPE EXAMPLES:
{swipe_ctx}

PRODUCT BRIEF:
- Product: {brief['product_name']}
- Audience: {brief['target_audience']}
- Pain: {brief['core_pain']}
- Mechanism: {brief['mechanism']}
- Proof: {brief['proof_metric']}
- CTA: {brief['cta']}

SECTION: {beat['stage']}
GOAL: {beat['goal']}
STRICT RULE: {beat['constraint']}
{avoid_repeat}
Additional Rules:
- 3 to 4 crisp sentences (45 to 70 words).
- Speak directly to "you" and "your".
- Never use "imagine", "unleash", "unlock", "seamless", or "elevate".
- Output ONLY the paragraph prose (no headings)."""

        prose = generate(prompt, max_new_tokens=200, temperature=0.6)
        drafted.append({"heading": beat["stage"], "prose": prose})
        prior_text += f"\n- {prose}"
        print(f"   ✓ {beat['stage']} ({len(prose.split())} words)")

    full = f"# Headline Placeholder\n\n" + "\n\n".join(f"## {d['heading']}\n\n{d['prose']}" for d in drafted)
    return {"sections": drafted, "full_markdown": full.strip()}

print("✓ Upgraded scanner & anti-repetition drafter ready.")

✓ Upgraded scanner & anti-repetition drafter ready.


In [87]:
def run_headline_tournament(brief):
    prompt = f"""Generate 3 distinct landing page headlines (under 12 words each) for {brief['product_name']}.
Audience: {brief['target_audience']}
Pain: {brief['core_pain']}
Mechanism: {brief['mechanism']}
Proof: {brief['proof_metric']}

Create one headline per psychological angle:
1. "pain": Calls out the frustrating daily bottleneck directly.
2. "outcome": Leads with the concrete time/money saved ({brief['proof_metric']}).
3. "contrarian": Challenges the broken way they currently handle this task.

Return ONLY valid JSON:
{{
  "variants": [
    {{"angle": "pain", "headline": "..."}},
    {{"angle": "outcome", "headline": "..."}},
    {{"angle": "contrarian", "headline": "..."}}
  ],
  "winner_index": <0, 1, or 2>,
  "why_it_wins": "one sentence explaining which stops a busy buyer from scrolling"
}}"""
    res = generate_json(prompt, max_new_tokens=400)
    variants = res.get("variants", [])
    idx = min(int(res.get("winner_index", 0)), len(variants) - 1)
    winner = variants[idx]["headline"]
    print("🏆 Headline Tournament:")
    for i, v in enumerate(variants):
        mark = "★" if i == idx else " "
        print(f"  {mark} [{v['angle'].upper()}] {v['headline']}")
    print(f"  Reason: {res.get('why_it_wins', '')}\n")
    return winner


def critique_as_buyer(markdown_text, brief, prior_issues=None):
    scan = scan_marketing_copy(markdown_text, brief)
    prior_block = ""
    if prior_issues:
        prior_block = "\nPRIOR ISSUES TO VERIFY:\n" + "\n".join(
            f"- {i.get('fix', i) if isinstance(i, dict) else i}" for i in prior_issues
        )

    prompt = f"""Roleplay as a busy, skeptical {brief['target_audience']}. Evaluate this landing page copy.

PRODUCT FACTS:
- Mechanism: {brief['mechanism']}
- Proof: {brief['proof_metric']}
- Approved CTA: {brief['cta']}

COPY TO EVALUATE:
{markdown_text}
{prior_block}

OBJECTIVE CHECKS:
- Buzzwords: {scan['hype_words'] or 'none'}
- Missing proof numbers: {scan['missing_proof_nums'] or 'none'}
- CTA leaked into early sections: {scan['cta_leaked_early']}
- Repeated phrases across sections: {scan['repeated_phrases'] or 'none'}

Think through the buyer's perspective first, then output ONLY valid JSON in this exact key order:
{{
  "step_by_step_reasoning": "2-3 plain-English sentences walking through: (1) whether the opening hook grabs attention, (2) whether the proof feels believable, and (3) what specific sentence still feels weak or repetitive.",
  "buyer_gut_reaction": "1 crisp sentence summarizing the buyer's honest reaction",
  "new_issues": [
    {{
      "section_heading": "exact H2 heading",
      "type": "vague_claim|repetition|flow|weak_hook",
      "evidence": "exact 4-10 word quote from the copy",
      "fix": "specific edit using only PRODUCT FACTS"
    }}
  ],
  "hook_strength": <1-3>,
  "believability_and_proof": <1-3>,
  "clarity_and_punch": <1-3>
}}

STRICT GRADING RULES:
- NEVER flag sentences under 14 words as wordy.
- NEVER flag the Approved CTA ("{brief['cta']}") or Proof numbers as hype.
- NEVER ask for videos, screenshots, or external case studies not in PRODUCT FACTS.
- Score 3 on each subscore if the section flows logically without repeating earlier sections, states the proof numbers factually, and uses crisp, concrete wording (leaving `new_issues` empty `[]`)."""

    critique = generate_json(prompt, max_new_tokens=720)

    valid_issues = []
    for i in critique.get("new_issues", []):
        ev = i.get("evidence", "").strip()
        if not ev or ev.lower() not in markdown_text.lower():
            continue
        if i.get("type") == "wordy" and len(ev.split()) < 15:
            continue
        if brief["cta"].lower() in ev.lower():
            continue
        valid_issues.append(i)

    for hw in scan["hype_words"]:
        valid_issues.append({
            "section_heading": "Global",
            "type": "hype",
            "evidence": hw,
            "fix": f"Remove '{hw}' and state the fact directly."
        })
    if scan["cta_leaked_early"]:
        valid_issues.append({
            "section_heading": "The Cost of Inaction & Mechanism",
            "type": "duplicate_cta",
            "evidence": "trial",
            "fix": "Remove the free trial / credit card CTA from this section—save the CTA exclusively for the final section."
        })
    if scan["missing_proof_nums"]:
        valid_issues.append({
            "section_heading": "Proof & Call to Action",
            "type": "missing_proof",
            "evidence": "",
            "fix": f"State the full proof metric factually: '{brief['proof_metric']}'."
        })
    for rep in scan["repeated_phrases"][:2]:
        valid_issues.append({
            "section_heading": "The Cost of Inaction & Mechanism",
            "type": "repetition",
            "evidence": rep,
            "fix": f"Avoid repeating '{rep}' from Section 1; focus on how the product works."
        })

    critique["new_issues"] = valid_issues
    raw_llm = (
        int(critique.get("hook_strength", 2))
        + int(critique.get("believability_and_proof", 2))
        + int(critique.get("clarity_and_punch", 2))
    )
    penalty = (
        len(scan["hype_words"]) * 2
        + (2 if scan["cta_leaked_early"] else 0)
        + (2 if scan["missing_proof_nums"] else 0)
        + min(2, len(scan["repeated_phrases"]))
    )
    bonus = 1 if (not valid_issues and raw_llm >= 8 and penalty == 0) else 0
    critique["score"] = max(1, min(10, raw_llm - penalty + bonus))
    critique["scan"] = scan
    return critique

def revise_marketing_copy(markdown_text, critique, brief):
    issues = critique.get("new_issues", [])
    if not issues:
        return markdown_text

    title_match = re.search(r"^#\s+(.+)$", markdown_text, flags=re.MULTILINE)
    title = title_match.group(1).strip() if title_match else brief["product_name"]
    parts = re.split(r"^##\s+(.+)$", markdown_text, flags=re.MULTILINE)

    rebuilt = f"# {title}\n\n"
    for i in range(1, len(parts), 2):
        heading, prose = parts[i].strip(), parts[i+1].strip()
        is_final_section = (i + 2 >= len(parts))

        sec_issues = [
            iss for iss in issues
            if (iss.get("evidence") and iss["evidence"].lower() in prose.lower())
            or iss.get("section_heading", "").lower() in (heading.lower(), "global")
        ]

        if not sec_issues:
            rebuilt += f"## {heading}\n\n{prose}\n\n"
            continue

        cta_rule = (
            f"End with the CTA: '{brief['cta']}'."
            if is_final_section
            else "Do NOT mention free trials, credit cards, or CTAs in this section."
        )
        fix_list = "\n".join(f"- {iss['fix']}" for iss in sec_issues)

        prompt = f"""Rewrite this landing page section to fix every editorial note.

PRODUCT FACTS:
- Product: {brief['product_name']}
- Mechanism: {brief['mechanism']}
- Proof: {brief['proof_metric']}

ORIGINAL SECTION ({heading}):
{prose}

FIXES REQUIRED:
{fix_list}

Rules:
- 3 to 4 crisp sentences (45 to 70 words).
- {cta_rule}
- Speak directly to "you/your". Zero hype words or "imagine".
- Output ONLY the revised paragraph."""

        new_prose = generate(prompt, max_new_tokens=200, temperature=0.45)
        new_prose = re.sub(r"^#+\s+.*$", "", new_prose, flags=re.MULTILINE).strip()
        rebuilt += f"## {heading}\n\n{new_prose}\n\n"

    return rebuilt.strip()

print("✓ Calibrated buyer critic & section-aware rewriter ready.")

✓ Calibrated buyer critic & section-aware rewriter ready.


In [75]:
import os
import json
import faiss
from datetime import datetime
from google.colab import drive

drive.mount('/content/drive')

PROJECT_DIR = "/content/drive/MyDrive/quill_LLM_runs"
os.makedirs(PROJECT_DIR, exist_ok=True)

def save_run_to_drive(final_result, index=None, style_chunks=None, save_dir=PROJECT_DIR):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    slug = re.sub(r"[^a-z0-9]+", "_", final_result["topic"].lower()).strip("_")[:40]
    run_folder = os.path.join(save_dir, f"{timestamp}_{slug}")
    os.makedirs(run_folder, exist_ok=True)

    # 1. Save winning copy as Markdown
    md_path = os.path.join(run_folder, "final_copy.md")
    with open(md_path, "w", encoding="utf-8") as f:
        f.write(final_result["final"])

    # 2. Save full round-by-round score and draft history as JSON
    log_path = os.path.join(run_folder, "critique_history.json")
    with open(log_path, "w", encoding="utf-8") as f:
        json.dump(final_result, f, indent=2, ensure_ascii=False)

    # 3. Save FAISS swipe-file index & chunks
    if index is not None and style_chunks is not None:
        faiss.write_index(index, os.path.join(run_folder, "swipe.index"))
        with open(os.path.join(run_folder, "swipe_chunks.json"), "w", encoding="utf-8") as f:
            json.dump(style_chunks, f, indent=2, ensure_ascii=False)

    print(f"✓ Saved run artifacts to:\n  {run_folder}")
    return run_folder

print(f"✓ Google Drive connected. Saving outputs to: {PROJECT_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Google Drive connected. Saving outputs to: /content/drive/MyDrive/quill_LLM_runs


In [82]:
def run_full_marketing_agent(brief, max_rounds=3, target_score=8):
    # 1. Pick the strongest headline angle via 3-way tournament
    winning_headline = run_headline_tournament(brief)

    # 2. Draft the PAS body sections
    draft_obj = draft_marketing_piece(brief)
    body_only = "\n\n".join(f"## {s['heading']}\n\n{s['prose']}" for s in draft_obj["sections"])
    draft = f"# {winning_headline}\n\n{body_only}"

    best_draft, best_score = draft, -1
    history, prior = [], None

    # 3. Skeptical buyer critique & surgical revision loop
    for r in range(1, max_rounds + 1):
        print(f"\n{'='*60}\n🔍 BUYER CRITIQUE ROUND {r}\n{'='*60}")
        c = critique_as_buyer(draft, brief, prior_issues=prior)
        s = c["scan"]
        print(f"Score: {c['score']}/10 | Hype: {len(s['hype_words'])} | "
              f"You/We: {s['you_count']}/{s['we_count']} | Avg Sentence: {s['avg_sentence_len']}w")
        print(f"Buyer Thought: \"{c.get('buyer_gut_reaction', '')}\"")

        for iss in c.get("new_issues", []):
            print(f"  • [{iss['type']}] \"{iss['evidence']}\" → {iss['fix']}")

        history.append({"round": r, "score": c["score"], "draft": draft})
        if c["score"] > best_score:
            best_score, best_draft = c["score"], draft

        if c["score"] >= target_score and not c["new_issues"]:
            print(f"\n✓ High-converting copy ready ({c['score']}/10)!")
            break

        if r < max_rounds:
            print("\n✏️  REVISING WEAK SPOTS...")
            draft = revise_marketing_copy(draft, c, brief)
            prior = c["new_issues"]

    print(f"\n📈 Score progression: {' → '.join(str(h['score']) for h in history)}")
    return {"topic": brief["product_name"], "final": best_draft, "history": history, "final_score": best_score}


# Swap in your own product/service details here
my_brief = {
    "product_name": "LedgerSnap",
    "target_audience": "freelance developers and solo agency owners",
    "core_pain": "losing billable hours and receipts across messy PDF invoices at tax time",
    "mechanism": "forwards any email receipt or PDF invoice to a private inbox and auto-syncs clean line items to your ledger in 5 seconds",
    "proof_metric": "saves 4.5 hours per month across 900+ independent consultants",
    "cta": "Start your free 14-day trial—no credit card needed"
}

result = run_full_marketing_agent(my_brief, max_rounds=3, target_score=8)

print("\n" + "="*60 + f"\n📣 FINAL COPY ({result['final_score']}/10)\n" + "="*60)
print(result["final"])

# Automatically save markdown, JSON history, and FAISS index to Drive
save_run_to_drive(result, index=swipe_index, style_chunks=SWIPE_FILE)

🏆 Headline Tournament:
    [PAIN] Stop wasting time on messy invoices.
  ★ [OUTCOME] Save 4.5 hours/month with LedgerSnap.
    [CONTRARIAN] Why manually track receipts when you can automate?
  Reason: It directly addresses the time savings, stopping busy buyers from scrolling.

🎯 Drafting PAS copy for: LedgerSnap
   ✓ The Hook & Problem (60 words)
   ✓ The Cost of Inaction & Mechanism (53 words)
   ✓ Proof & Call to Action (29 words)

🔍 BUYER CRITIQUE ROUND 1
Score: 4/10 | Hype: 0 | You/We: 14/0 | Avg Sentence: 12.2w
Buyer Thought: "The copy feels a bit vague and repetitive. It could use clearer benefits and a stronger hook."
  • [vague_claim] "But then, weeks later, you're sifting through mountains of PDFs, trying to piece together what was sent where." → But then, weeks later, you're wasting time sifting through PDFs, losing track of what was sent.
  • [repetition] "With LedgerSnap, every email receipt or PDF invoice lands in a dedicated, organized inbox. Within seconds, all the rele

'/content/drive/MyDrive/quill_LLM_runs/20261112_223205_ledgersnap'

In [89]:
!pip install -q gradio fpdf2

In [90]:
import os
import re
from datetime import datetime
import gradio as gr
from fpdf import FPDF


def build_copy_pdf(brief, final_markdown, history, best_score):
    """Generates a clean monochrome PDF containing the final copy and review notes."""
    os.makedirs("exports", exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    slug = re.sub(r"[^a-z0-9]+", "_", brief["product_name"].lower()).strip("_")[:30] or "copy"
    pdf_path = os.path.join("exports", f"{slug}_{timestamp}.pdf")

    pdf = FPDF()
    pdf.set_auto_page_break(auto=True, margin=18)
    pdf.add_page()

    # Use Colab's pre-installed DejaVu fonts for full Unicode support (em-dashes, smart quotes)
    sans_path = "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"
    sans_bold_path = "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf"
    mono_path = "/usr/share/fonts/truetype/dejavu/DejaVuSansMono.ttf"

    has_unicode_fonts = all(os.path.exists(p) for p in (sans_path, sans_bold_path, mono_path))
    if has_unicode_fonts:
        pdf.add_font("Sans", "", sans_path, uni=True)
        pdf.add_font("Sans", "B", sans_bold_path, uni=True)
        pdf.add_font("Mono", "", mono_path, uni=True)
        font_sans, font_mono = "Sans", "Mono"
    else:
        font_sans, font_mono = "Helvetica", "Courier"

    def clean_txt(t):
        if has_unicode_fonts:
            return t
        return (
            t.replace("—", "--").replace("–", "-")
             .replace("’", "'").replace("‘", "'")
             .replace("“", '"').replace("”", '"')
             .encode("latin-1", "replace").decode("latin-1")
        )

    # Top Kicker & Divider
    pdf.set_font(font_mono, "", 8.5)
    pdf.set_text_color(90, 90, 95)
    pdf.cell(0, 6, clean_txt(f"MARKET-COPY-LLM // {brief['product_name'].upper()} // QUALITY SCORE: {best_score}/10"), ln=True)
    pdf.set_draw_color(15, 15, 15)
    pdf.set_line_width(0.4)
    pdf.line(10, pdf.get_y() + 1, 200, pdf.get_y() + 1)
    pdf.ln(7)

    # Parse and render Final Copy Markdown
    for line in final_markdown.strip().splitlines():
        line = line.strip()
        if not line:
            pdf.ln(3)
            continue
        if line.startswith("# "):
            pdf.set_font(font_sans, "B", 17)
            pdf.set_text_color(10, 10, 10)
            pdf.multi_cell(0, 8, clean_txt(line[2:].strip()))
            pdf.ln(3)
        elif line.startswith("## "):
            pdf.ln(2)
            pdf.set_font(font_mono, "", 9.5)
            pdf.set_text_color(60, 60, 65)
            pdf.multi_cell(0, 6, clean_txt(line[3:].strip().upper()))
            pdf.ln(1)
        else:
            pdf.set_font(font_sans, "", 10.5)
            pdf.set_text_color(25, 25, 25)
            pdf.multi_cell(0, 6, clean_txt(line))

    # Review Trajectory Footer Block
    pdf.ln(8)
    pdf.set_draw_color(210, 210, 215)
    pdf.line(10, pdf.get_y(), 200, pdf.get_y())
    pdf.ln(5)

    progression = " -> ".join(f"{h['score']}/10" for h in history)
    pdf.set_font(font_mono, "", 8.5)
    pdf.set_text_color(80, 80, 85)
    pdf.cell(0, 5, clean_txt(f"AUDIENCE: {brief['target_audience']}"), ln=True)
    pdf.cell(0, 5, clean_txt(f"SCORE HISTORY: {progression} (BEST: {best_score}/10)"), ln=True)

    pdf.output(pdf_path)
    return pdf_path


def generate_copy_with_ui(
    product_name,
    target_audience,
    core_pain,
    mechanism,
    proof_metric,
    cta,
    max_rounds,
    target_score
):
    if not product_name.strip() or not mechanism.strip():
        yield (
            "Please enter at least a **Product Name** and **How It Works** to begin.",
            "Waiting for product details.",
            gr.update(visible=False)
        )
        return

    brief = {
        "product_name": product_name.strip(),
        "target_audience": target_audience.strip(),
        "core_pain": core_pain.strip(),
        "mechanism": mechanism.strip(),
        "proof_metric": proof_metric.strip(),
        "cta": cta.strip(),
    }

    log_lines = [f"### STARTED — `{brief['product_name'].upper()}`\n"]
    yield "*Step 1 of 3: Testing 3 headline ideas...*", "\n".join(log_lines), gr.update(visible=False)

    # 1. Headline Tournament
    winning_headline = run_headline_tournament(brief)
    log_lines.append(f"**WINNING HEADLINE:** `{winning_headline}`\n")
    yield f"# {winning_headline}\n\n*Step 2 of 3: Writing the main sections...*", "\n".join(log_lines), gr.update(visible=False)

    # 2. Initial Draft
    draft_obj = draft_marketing_piece(brief)
    body_only = "\n\n".join(f"## {s['heading']}\n\n{s['prose']}" for s in draft_obj["sections"])
    draft = f"# {winning_headline}\n\n{body_only}"

    best_draft, best_score = draft, -1
    history, prior = [], None
    log_lines.append("**FIRST DRAFT:** Complete. Starting customer review.\n\n---\n")
    yield draft, "\n".join(log_lines), gr.update(visible=False)

    # 3. Review & Revision Loop
    for r in range(1, int(max_rounds) + 1):
        log_lines.append(f"#### REVIEW ROUND {r:02d}")
        yield best_draft if best_score > 0 else draft, "\n".join(log_lines), gr.update(visible=False)

        c = critique_as_buyer(draft, brief, prior_issues=prior)
        s = c["scan"]
        score = c["score"]

        log_lines.append(
            f"- **Quality Score:** `{score}/10` | **Buzzwords:** `{len(s['hype_words'])}` | "
            f"**Reader vs. Brand Focus:** `{s['you_count']}:{s['we_count']}` | **Avg. Sentence Length:** `{s['avg_sentence_len']} words`"
        )
        if c.get("step_by_step_reasoning"):
            log_lines.append(f"- **Agent Reasoning:** {c['step_by_step_reasoning']}")
        log_lines.append(f"- **Customer Feedback:** *\"{c.get('buyer_gut_reaction', '')}\"*")

        issues = c.get("new_issues", [])
        if issues:
            for iss in issues:
                log_lines.append(f"  - **To Fix:** {iss['fix']}")
        else:
            log_lines.append("  - **Passed:** Clear, persuasive, and free of clichés.")

        history.append({"round": r, "score": score, "draft": draft})
        if score > best_score:
            best_score, best_draft = score, draft

        if score >= int(target_score) and not issues:
            log_lines.append(f"\n**COMPLETE:** Reached your quality goal (`{target_score}/10`).")
            break

        if r < int(max_rounds):
            log_lines.append("\n*Rewriting sections that need improvement...*\n")
            yield best_draft, "\n".join(log_lines), gr.update(visible=False)
            draft = revise_marketing_copy(draft, c, brief)
            prior = issues

    progression = " -> ".join(f"{h['score']}/10" for h in history)
    log_lines.append(f"\n---\n### SCORE HISTORY: `{progression}` (BEST: `{best_score}/10`)")

    # 4. Build PDF and reveal Download Button
    pdf_file = build_copy_pdf(brief, best_draft, history, best_score)
    log_lines.append("\n**PDF READY:** Click **DOWNLOAD PDF** above to save a copy.")

    yield best_draft, "\n".join(log_lines), gr.update(value=pdf_file, visible=True)


PRESET_BRIEFS = [
    [
        "LedgerSnap",
        "freelance developers and solo agency owners",
        "losing billable hours and receipts across messy PDF invoices at tax time",
        "forwards any email receipt or PDF invoice to a private inbox and auto-syncs clean line items to your ledger in 5 seconds",
        "saves 4.5 hours per month across 900+ independent consultants",
        "Start your free 14-day trial—no credit card needed",
        3, 8
    ],
    [
        "DocuPulse",
        "B2B SaaS account executives",
        "sending proposals into a black hole and guessing when to follow up",
        "alerts you on Slack the exact second a prospect re-opens your pricing page or forwards your deck to a decision-maker",
        "increases proposal close rates by 27% across 450+ sales teams",
        "Connect your inbox free in 60 seconds",
        3, 8
    ],
    [
        "QueryGuard",
        "backend engineers and startup CTOs",
        "waking up to 3 AM database outages caused by unindexed N+1 queries",
        "analyzes pull requests in CI/CD and blocks slow SQL migrations before they ever hit production",
        "cuts p99 API latency by 62% with zero production incidents",
        "Install the GitHub App free—takes 2 minutes",
        3, 8
    ]
]

CUSTOM_CSS = """
@import url('https://fonts.googleapis.com/css2?family=IBM+Plex+Mono:wght@400;500;600&family=IBM+Plex+Sans:wght@400;500;600&family=JetBrains+Mono:wght@400;500;600&display=swap');

:root, .gradio-container {
    --font: 'IBM Plex Sans', -apple-system, BlinkMacSystemFont, sans-serif !important;
    --font-mono: 'JetBrains Mono', 'IBM Plex Mono', monospace !important;
    --background-fill-primary: #FFFFFF !important;
    --background-fill-secondary: #FAFAFA !important;
    --border-color-primary: #E4E4E7 !important;
    --border-color-accent: #09090B !important;
    --color-accent: #09090B !important;
    --body-text-color: #09090B !important;
    --body-text-color-subdued: #52525B !important;
    --block-radius: 2px !important;
    --input-radius: 2px !important;
    --button-large-radius: 2px !important;
}

.gradio-container {
    max-width: 1240px !important;
    margin: 0 auto !important;
    padding: 28px 24px !important;
    background-color: #FFFFFF !important;
}

.app-header {
    border-bottom: 1px solid #09090B;
    padding-bottom: 16px;
    margin-bottom: 24px;
}
.app-kicker {
    font-family: 'JetBrains Mono', monospace;
    font-size: 11px;
    letter-spacing: 0.08em;
    text-transform: uppercase;
    color: #52525B;
    margin-bottom: 4px;
}
.app-title {
    font-family: 'IBM Plex Sans', sans-serif;
    font-size: 24px;
    font-weight: 600;
    letter-spacing: -0.02em;
    color: #09090B;
    margin: 0;
}
.app-subtitle {
    font-family: 'IBM Plex Sans', sans-serif;
    font-size: 13.5px;
    color: #52525B;
    margin-top: 4px;
}

.section-tag {
    font-family: 'JetBrains Mono', monospace;
    font-size: 11px;
    font-weight: 600;
    letter-spacing: 0.06em;
    text-transform: uppercase;
    color: #09090B;
    border-bottom: 1px solid #E4E4E7;
    padding-bottom: 6px;
    margin-bottom: 12px;
}

label span, .gr-block label {
    font-family: 'IBM Plex Mono', monospace !important;
    font-size: 11.5px !important;
    font-weight: 500 !important;
    text-transform: uppercase !important;
    letter-spacing: 0.04em !important;
    color: #27272A !important;
}

textarea, input[type="text"], input[type="number"] {
    font-family: 'IBM Plex Sans', sans-serif !important;
    font-size: 13.5px !important;
    background-color: #FAFAFA !important;
    border: 1px solid #D4D4D8 !important;
    border-radius: 2px !important;
    color: #09090B !important;
    box-shadow: none !important;
}

textarea:focus, input[type="text"]:focus {
    border-color: #09090B !important;
    background-color: #FFFFFF !important;
}

/* Primary Monochrome Button */
button.primary {
    background: #09090B !important;
    color: #FFFFFF !important;
    border: 1px solid #09090B !important;
    border-radius: 2px !important;
    font-family: 'JetBrains Mono', monospace !important;
    font-size: 12px !important;
    font-weight: 500 !important;
    letter-spacing: 0.05em !important;
    text-transform: uppercase !important;
    padding: 10px 16px !important;
    box-shadow: none !important;
    transition: background 0.15s ease !important;
}
button.primary:hover {
    background: #27272A !important;
}

/* Secondary Monochrome Download Button */
button.secondary, a.download-btn, .download-btn button {
    background: #FFFFFF !important;
    color: #09090B !important;
    border: 1px solid #09090B !important;
    border-radius: 2px !important;
    font-family: 'JetBrains Mono', monospace !important;
    font-size: 12px !important;
    font-weight: 500 !important;
    letter-spacing: 0.05em !important;
    text-transform: uppercase !important;
    padding: 10px 16px !important;
    box-shadow: none !important;
}
button.secondary:hover, .download-btn button:hover {
    background: #F4F4F5 !important;
}

.tab-nav {
    border-bottom: 1px solid #E4E4E7 !important;
}
.tab-nav button {
    font-family: 'JetBrains Mono', monospace !important;
    font-size: 12px !important;
    text-transform: uppercase !important;
    letter-spacing: 0.04em !important;
    color: #71717A !important;
    border-radius: 0 !important;
    border: none !important;
    border-bottom: 2px solid transparent !important;
    padding: 8px 14px !important;
}
.tab-nav button.selected {
    color: #09090B !important;
    font-weight: 600 !important;
    border-bottom: 2px solid #09090B !important;
    background: transparent !important;
}

.output-panel {
    background: #FAFAFA !important;
    border: 1px solid #E4E4E7 !important;
    border-radius: 2px !important;
    padding: 20px !important;
    min-height: 460px;
}
.output-panel h1 {
    font-family: 'IBM Plex Sans', sans-serif !important;
    font-size: 21px !important;
    font-weight: 600 !important;
    letter-spacing: -0.015em !important;
    color: #09090B !important;
    border-bottom: 1px solid #E4E4E7;
    padding-bottom: 10px;
}
.output-panel h2 {
    font-family: 'JetBrains Mono', monospace !important;
    font-size: 12px !important;
    font-weight: 600 !important;
    text-transform: uppercase !important;
    letter-spacing: 0.05em !important;
    color: #3F3F46 !important;
    margin-top: 18px !important;
}
.output-panel p, .output-panel li {
    font-family: 'IBM Plex Sans', sans-serif !important;
    font-size: 14px !important;
    line-height: 1.65 !important;
    color: #18181B !important;
}
.output-panel code {
    font-family: 'JetBrains Mono', monospace !important;
    font-size: 12px !important;
    background: #EFEFEF !important;
    color: #09090B !important;
    padding: 2px 5px !important;
    border-radius: 2px !important;
}

table {
    font-family: 'IBM Plex Sans', sans-serif !important;
    font-size: 12.5px !important;
    border: 1px solid #E4E4E7 !important;
}
th {
    font-family: 'JetBrains Mono', monospace !important;
    font-size: 11px !important;
    text-transform: uppercase !important;
    background: #F4F4F5 !important;
    color: #09090B !important;
}
"""

theme = gr.themes.Base(
    primary_hue="neutral",
    secondary_hue="neutral",
    neutral_hue="neutral",
    radius_size="none",
)

with gr.Blocks(theme=theme, css=CUSTOM_CSS, title="Market-Copy-LLM") as demo:
    gr.HTML("""
        <div class="app-header">
            <div class="app-kicker">MARKET-COPY-LLM // COPYWRITING STUDIO</div>
            <h1 class="app-title">Self-Improving Marketing Copywriter</h1>
            <div class="app-subtitle">Tests multiple headline angles, writes persuasive landing page copy, and polishes weak sections through simulated customer reviews.</div>
        </div>
    """)

    with gr.Row(equal_height=False):
        with gr.Column(scale=5):
            gr.HTML('<div class="section-tag">01 // PRODUCT DETAILS</div>')
            product_name = gr.Textbox(label="Product Name", placeholder="e.g., LedgerSnap")
            target_audience = gr.Textbox(label="Target Audience", placeholder="e.g., freelance developers and solo agency owners")
            core_pain = gr.Textbox(label="Customer Problem", lines=2, placeholder="What daily frustration does your product solve?")
            mechanism = gr.Textbox(label="How It Works", lines=2, placeholder="Explain simply how your product solves the problem.")
            proof_metric = gr.Textbox(label="Results & Proof", placeholder="e.g., saves 4.5 hours per month across 900+ consultants")
            cta = gr.Textbox(label="Call to Action", placeholder="e.g., Start your free 14-day trial—no credit card needed")

            with gr.Row():
                max_rounds = gr.Slider(minimum=1, maximum=4, value=3, step=1, label="Revision Rounds")
                target_score = gr.Slider(minimum=6, maximum=10, value=8, step=1, label="Quality Goal")

            with gr.Row():
                run_btn = gr.Button("WRITE & REFINE COPY", variant="primary")
                download_btn = gr.DownloadButton("DOWNLOAD PDF", variant="secondary", visible=False, elem_classes=["download-btn"])

        with gr.Column(scale=6):
            gr.HTML('<div class="section-tag">02 // RESULTS</div>')
            with gr.Tabs():
                with gr.TabItem("FINAL COPY"):
                    final_output = gr.Markdown(
                        "*Fill in your product details on the left and click **WRITE & REFINE COPY**.*",
                        elem_classes=["output-panel"]
                    )
                with gr.TabItem("REVIEW NOTES"):
                    critique_log = gr.Markdown(
                        "*Round-by-round quality scores, agent reasoning, customer feedback, and edits will appear here.*",
                        elem_classes=["output-panel"]
                    )

    gr.HTML('<div class="section-tag" style="margin-top: 24px;">03 // SAMPLE PRODUCTS (CLICK TO TRY)</div>')
    gr.Examples(
        examples=PRESET_BRIEFS,
        inputs=[product_name, target_audience, core_pain, mechanism, proof_metric, cta, max_rounds, target_score],
        label=""
    )

    run_btn.click(
        fn=generate_copy_with_ui,
        inputs=[product_name, target_audience, core_pain, mechanism, proof_metric, cta, max_rounds, target_score],
        outputs=[final_output, critique_log, download_btn]
    )

demo.queue().launch(share=True, debug=False)

/tmp/ipykernel_1106/1921427553.py:421: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=theme, css=CUSTOM_CSS, title="Market-Copy-LLM") as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c2d7bd01ec2d28f50d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
